# Parameters and configuration

In [0]:
from pyspark.sql import functions as F

from src.ingestion.provenance import build_provenance_columns, build_run_id

CATALOG = "trippilot_dev"
BRONZE_SCHEMA = "bronze"

SOURCE = "insideairbnb"
CITY = "toronto"
SNAPSHOT_DATE = "2026-09-16"
RAW_PATH = (
    "/Volumes/trippilot_dev/bronze/source_files/"
    "insideairbnb/toronto/listings.csv.gz"
)

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airbnb_listings_bronze"

In [0]:
run_id = build_run_id()

provenance = build_provenance_columns(
    source=SOURCE,
    city=CITY,
    snapshot_date=SNAPSHOT_DATE,
    raw_path=RAW_PATH,
    run_id=run_id,
)

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Run ID: {run_id}")
print(provenance)

In [0]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False) #Keep inferSchema=False in Bronze to retain the source representation and defer type decisions to Silver. This avoids silently turning malformed prices, dates, or numbers into nulls during ingestion.
    .option("multiLine", True)
    .option("escape", '"')
    .csv(RAW_PATH)
)

print(f"Raw rows read: {raw_df.count()}")
display(raw_df.limit(5))

In [0]:
bronze_df = (
    raw_df
    .withColumn("_source", F.lit(provenance["source"]))
    .withColumn("_city", F.lit(provenance["city"]))
    .withColumn("_snapshot_date", F.lit(provenance["snapshot_date"]))
    .withColumn("_raw_path", F.lit(provenance["raw_path"]))
    .withColumn("_run_id", F.lit(provenance["run_id"]))
    .withColumn("_ingested_at_utc", F.lit(provenance["ingested_at_utc"]))
)

display(bronze_df.limit(5))

# Write Bronze to an isolated test table first
Before writing the real table, verify the logic with a test table:

In [0]:
TEST_BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airbnb_listings_bronze_test"

(
    bronze_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TEST_BRONZE_TABLE)
)

print(f"Test Bronze table written: {TEST_BRONZE_TABLE}")
print(f"Test rows written: {spark.table(TEST_BRONZE_TABLE).count()}")

# Write the real Bronze table

In [0]:
(
    bronze_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze table written: {BRONZE_TABLE}")
print(f"Bronze rows written: {spark.table(BRONZE_TABLE).count()}")

so far, overwrite is acceptable for this one-city proof because the table contains only Toronto. When we have mutiple city, we will use safer current-city replacement and idempotent rerun semantics.

# Verify the output

In [0]:
bronze_table = spark.table(BRONZE_TABLE)

verification = bronze_table.agg(
    F.count("*").alias("bronze_row_count"),
    F.countDistinct("_run_id").alias("distinct_run_ids"),
    F.countDistinct("_city").alias("distinct_cities"),
    F.countDistinct("_snapshot_date").alias("distinct_snapshot_dates"),
    F.count(F.when(F.col("id").isNull(), 1)).alias("null_listing_ids"),
)

display(verification)

display(
    bronze_table
    .select(
        "id",
        "_source",
        "_city",
        "_snapshot_date",
        "_raw_path",
        "_run_id",
        "_ingested_at_utc",
    )
    .limit(5)
)